In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import os
import cv2
import psutil
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from scikeras.wrappers import KerasClassifier


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify that the drive is mounted
!ls /content/drive/MyDrive/Germ\ DS/

# Check if GPU is available
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

# Function to get current memory usage
def get_memory_usage():
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()
    return mem_info.rss / (1024 ** 3)  # Convert bytes to GB

print(f"Current memory usage: {get_memory_usage()} GB")

# Setting a random seed for reproducibility
np.random.seed(241030)


In [ ]:
# Paths to the CSV files and image directories
train_csv_path = "/content/drive/MyDrive/Germ DS/Train.csv"
meta_csv_path = "/content/drive/MyDrive/Germ DS/Meta.csv"
test_csv_path = "/content/drive/MyDrive/Germ DS/Test.csv"
base_img_dir = "/content/drive/MyDrive/Germ DS/"

# Load CSV files
train_df = pd.read_csv(train_csv_path)
meta_df = pd.read_csv(meta_csv_path)
test_df = pd.read_csv(test_csv_path)

# Display the columns to verify
print("Train DataFrame Columns:", train_df.columns)
print("Meta DataFrame Columns:", meta_df.columns)
print("Test DataFrame Columns:", test_df.columns)

# Check the balanced dataset
print(train_df['ClassId'].value_counts())


In [ ]:
# Function to load images and resize them
def load_image(path):
    image = cv2.imread(path)
    if image is None:
        print(f"Failed to read image: {path}")
        return None
    resized_image = cv2.resize(image, (128, 128))
    return resized_image

# Function to load and duplicate training images without cropping
def load_and_duplicate_train_images(base_dir, df, max_images_per_class=160):
    images = []
    labels = []
    count = 0
    loaded_files = set()
    class_counts = {}  # Dictionary to track the number of images loaded per class

    for index, row in df.iterrows():
        class_id = row['ClassId']
        img_dir = os.path.join(base_dir, str(class_id))

        # Check if the directory exists
        if not os.path.isdir(img_dir):
            print(f"Directory does not exist: {img_dir}")
            continue  # Skip to the next iteration if the directory doesn't exist

        img_name = os.path.basename(row['Path'])
        img_path = os.path.join(img_dir, img_name)

        # Limit the number of images per class
        if class_counts.get(class_id, 0) >= max_images_per_class:
            continue  # Skip loading more images for this class

        if os.path.isfile(img_path) and img_path not in loaded_files:
            loaded_files.add(img_path)  # Avoid duplicate loading

            # Load the image
            image = load_image(img_path)
            if image is not None:
                images.append(image)
                labels.append(class_id)
                # Duplicate the image and label to increase dataset size
                images.append(image)
                labels.append(class_id)

                # Track the number of images loaded per class
                class_counts[class_id] = class_counts.get(class_id, 0) + 1

            count += 1

            # Print progress every 200 images
            if count % 200 == 0:
                print(f"{count} training images loaded (duplicates included)")

    return np.array(images), np.array(labels)


In [ ]:
# Load the training images with duplication
train_images, train_labels = load_and_duplicate_train_images("/content/drive/MyDrive/Germ DS/Train", train_df)

# Check if images and labels are loaded correctly
if len(train_images) == 0 or len(train_labels) == 0:
    print("No images or labels loaded. Please check the dataset and paths.")
else:
    print(f"Loaded {len(train_images)} images (including duplicates).")

# Normalize the images
train_images = train_images / 255.0

# Split the data into training and validation sets
train_images, val_images, train_labels, val_labels = train_test_split(train_images, train_labels, test_size=0.2, random_state=241030)

# Check shapes of the datasets
print("Training data shape:", train_images.shape, train_labels.shape)
print("Validation data shape:", val_images.shape, val_labels.shape)


In [ ]:
# Function to load a maximum of 50 images per class from the test set
def load_limited_test_images(base_dir, df, max_images_per_class=50):
    images = []
    labels = []
    class_count = {}

    for index, row in df.iterrows():
        class_id = row['ClassId']
        img_name = os.path.basename(row['Path'])
        img_path = os.path.join(base_dir, img_name)

        if os.path.isfile(img_path):
            if class_count.get(class_id, 0) < max_images_per_class:
                image = load_image(img_path)
                if image is not None:
                    images.append(image)
                    labels.append(class_id)
                    class_count[class_id] = class_count.get(class_id, 0) + 1

    return np.array(images), np.array(labels)

# Load the test images with a limit of 50 images per class
test_images, test_labels = load_limited_test_images("/content/drive/MyDrive/Germ DS/Test", test_df, max_images_per_class=50)

# Check if test images and labels are loaded correctly
if len(test_images) == 0 or len(test_labels) == 0:
    print("No test images or labels loaded. Please check the dataset and paths.")
else:
    print(f"Loaded {len(test_images)} test images (limited to 50 per class).")

# Normalize the test images
test_images = test_images / 255.0


In [ ]:
# Define the model architecture
def create_model():
    model = Sequential([
        Conv992D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Conv2D(64, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Conv2D(128, (3, 3), activation='relu'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Flatten(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(len(np.unique(train_labels)), activation='softmax')
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Instantiate the model
model_instance = create_model()






In [ ]:
# Data Augmentation
datagen = ImageDataGenerator(zoom_range=0.2)

# Train and validation generators
train_generator = datagen.flow(train_images, train_labels, batch_size=32)
val_generator = ImageDataGenerator().flow(val_images, val_labels, batch_size=32)

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6)

In [ ]:
# !pip install --upgrade tensorflow


In [ ]:
# Train the model
history = model_instance.fit(
    train_generator,
    validation_data=val_generator,
    epochs=20,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

In [ ]:
# Evaluate on the training set
train_loss, train_acc = model_instance.evaluate(train_images, train_labels, verbose=1)
print(f"Training accuracy: {train_acc}")

# Evaluate on the validation set
val_loss, val_acc = model_instance.evaluate(val_images, val_labels, verbose=1)
print(f"Validation accuracy: {val_acc}")

# Evaluate on the test set
test_loss, test_acc = model_instance.evaluate(test_images, test_labels, verbose=1)
print(f"Test accuracy: {test_acc}")


In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV

# Define a function to create the model for grid search
def create_model(optimizer='adam', learning_rate=0.001):
    if optimizer == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer == 'sgd':
        optimizer = SGD(learning_rate=learning_rate, momentum=0.9)

    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', kernel_regularizer=l2(0.001), input_shape=(128, 128, 3)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Conv2D(64, (3, 3), activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Conv2D(128, (3, 3), activation='relu', kernel_regularizer=l2(0.001)),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.3),

        Flatten(),
        Dense(512, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        Dense(len(np.unique(train_labels)), activation='softmax')
    ])

    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Wrap the Keras model in KerasClassifier
model = KerasClassifier(
    build_fn=create_model,
    optimizer='adam',       # Default optimizer
    learning_rate=0.001,     # Default learning rate
    verbose=0
)

# Define the hyperparameter grid for batch_size and epochs
param_grid = {
    'batch_size': [32, 64],
    'epochs': [10, 20]
}

# Set up GridSearchCV
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, verbose=1)

# Perform the grid search
grid_result = grid.fit(train_images, train_labels)

# Print the best hyperparameters and best score
print(f"Best Hyperparameters: {grid_result.best_params_}")
print(f"Best Accuracy: {grid_result.best_score_}")

# Retrieve the best model from GridSearchCV
best_model = grid_result.best_estimator_.model

# Evaluate the best model on the test set
test_loss, test_acc = best_model.evaluate(test_images, test_labels, verbose=1)
print(f"Test accuracy of the best model from grid search: {test_acc}")


In [ ]:
# Retrieve the best model from GridSearchCV
best_model_wrapper = grid_result.best_estimator_

# Unwrap the Keras model from the SciKeras wrapper
best_model = best_model_wrapper.model_

# Evaluate the best model on the test set
test_loss, test_acc = best_model.evaluate(test_images, test_labels, verbose=1)
print(f"Test accuracy of the best model from grid search: {test_acc}")


In [ ]:
# Display the model summary
best_model.summary()

# Evaluate the best model on the test set
test_loss, test_acc = best_model.evaluate(test_images, test_labels, verbose=1)
print(f"Test accuracy of the best model from grid search: {test_acc}")


In [ ]:
import matplotlib.pyplot as plt

# Function to plot the training history
def plot_training_history(history):
    # Plot training & validation accuracy values
    plt.figure(figsize=(12, 4))

    # Accuracy plot
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss plot
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    # Display the plots
    plt.show()

# Call the function to plot training history
plot_training_history(history)


In [ ]:
class_id_to_sign_name = {
    0: "Speed limit (20km/h)",
    1: "Speed limit (30km/h)",
    2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)",
    4: "Speed limit (70km/h)",
    5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)",
    7: "Speed limit (100km/h)",
    8: "Speed limit (120km/h)",
    9: "No passing",
    10: "No passing for vehicles over 3.5 metric tons",
    11: "Right-of-way at the next intersection",
    12: "Priority road",
    13: "Yield",
    14: "Stop",
    15: "No vehicles",
    16: "Vehicles over 3.5 metric tons prohibited",
    17: "No entry",
    18: "General caution",
    19: "Dangerous curve to the left",
    20: "Dangerous curve to the right",
    21: "Double curve",
    22: "Bumpy road",
    23: "Slippery road",
    24: "Road narrows on the right",
    25: "Road work",
    26: "Traffic signals",
    27: "Pedestrians",
    28: "Children crossing",
    29: "Bicycles crossing",
    30: "Beware of ice/snow",
    31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",
    34: "Turn left ahead",
    35: "Ahead only",
    36: "Go straight or right",
    37: "Go straight or left",
    38: "Keep right",
    39: "Keep left",
    40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 metric tons"
}

# Convert the dictionary to a DataFrame
key_df = pd.DataFrame(list(class_id_to_sign_name.items()), columns=['ClassId', 'SignName'])



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow

In [ ]:
# Path to the folder containing images
folder_path = '/content/drive/MyDrive/Germ DS/MIXED SIGNS'

# Process each image in the folder
for image_name in os.listdir(folder_path):
    image_path = os.path.join(folder_path, image_name)
    image = cv2.imread(image_path)

    if image is not None:
        # Resize the image to 128x128 as expected by the model
        resized_image = cv2.resize(image, (128, 128))
        # Normalize the image
        normalized_image = resized_image / 255.0
        # Expand dimensions to match model input
        input_image = np.expand_dims(normalized_image, axis=0)

        # Make prediction using the trained model
        predictions = model_instance.predict(input_image)
        predicted_class = np.argmax(predictions)
        sign_name = class_id_to_sign_name.get(predicted_class, "Unknown Sign")

        # Display the image
        cv2_imshow(image)

        # Print the prediction
        print(f"Image: {image_name}, Predicted Sign: {sign_name}")
    else:
        print(f"Error reading image: {image_name}")

In [ ]:

# Use the model to predict on the test images
predicted_probs = model_instance.predict(test_images)
predicted_classes = np.argmax(predicted_probs, axis=1)  # Get the class with the highest probability

# Handle the case where test_labels are already in integer form
if len(test_labels.shape) == 1:  # Check if test_labels is a 1D array
    actual_classes = test_labels
else:  # If test_labels is one-hot encoded
    actual_classes = np.argmax(test_labels, axis=1)

# Map the actual and predicted classes to their corresponding sign names
actual_sign_names = [class_id_to_sign_name[class_id] for class_id in actual_classes]
predicted_sign_names = [class_id_to_sign_name[class_id] for class_id in predicted_classes]

# Create a DataFrame to store the results
results_df = pd.DataFrame({
    'Image': range(len(test_images)),  # You can modify this to include image file names if available
    'Actual Class': actual_classes,
    'Actual Sign': actual_sign_names,
    'Predicted Class': predicted_classes,
    'Predicted Sign': predicted_sign_names
})

# Save the DataFrame to a CSV file
results_df.to_csv('/content/drive/MyDrive/results.csv', index=False)

# Print the first few rows of the results for verification
print(results_df.head())